# Data preprocessing

Run from any directory. This notebook cleans `data/raw/city_day.csv` into `data/interim/city_day_clean.csv`.


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/raw/city_day.csv").exists())
raw_path = ROOT / "data/raw/city_day.csv"
output_path = ROOT / "data/interim/city_day_clean.csv"
df = pd.read_csv(raw_path)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.sort_values(["City", "Date"]).copy()


In [ ]:
pollutants = [c for c in ["PM2.5", "PM10", "NO", "NO2", "NOx", "NH3", "CO", "SO2", "O3", "Benzene", "Toluene", "Xylene"] if c in df]
if pollutants:
    df[pollutants] = df.groupby("City", sort=False)[pollutants].transform(lambda values: values.interpolate(limit_direction="both"))
    for column in pollutants:
        df[column] = df[column].fillna(df.groupby("City")[column].transform("median"))
if "AQI" in df:
    df = df.dropna(subset=["AQI"])
df = df.drop_duplicates()
if {"City", "Date"}.issubset(df.columns):
    df = df.drop_duplicates(["City", "Date"], keep="first")
for column in pollutants + (["AQI"] if "AQI" in df else []):
    lower, upper = df[column].quantile([0.01, 0.99])
    df[column] = df[column].clip(lower, upper)
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Saved {len(df):,} cleaned rows to {output_path}")
df.head()
